# The genetic-algorithm attack — evolution instead of gradients

A different attack philosophy: no gradients at all. A population of candidate perturbations **evolves** — selection by fitness (how benign the detector finds the result, minus a size penalty), crossover, and mutation restricted to the features the manipulation rules allow to change. This notebook follows the attack's three versions: the first build, the mimicry-seeded second version, and the developed third version, which guides evolution with RandomForest feature importances and seeds the population from the *nearest benign samples*.

**Where this fits:** Section 03 attacks the detectors built in `02-detectors/`, under the manipulation rules from `01-data-and-features/manipulation_maps.ipynb`. The defended models these attacks are later measured against live in `04-defenses/`, and the final, organised measurements are in `06-thesis-evaluation/`.

*Consolidated from `genetic_algorithm.ipynb`, `genetic_algorithm2.ipynb` and `genetic_algorithm3.ipynb` (originally in `a_malware` and `a_malware2`). The second version's tail — a pasted copy of the PGD classes — is not repeated; it lives in `pgd_and_adversarial_training_first_codebase.ipynb`. Outputs cleared.*

## Step 1 — Setup: the first codebase and its trained detector

In [ ]:
!pip install androguard
!pip install captum

import androguard
import captum

In [ ]:
import time
import os.path as path
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os
import random

torch.manual_seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

!git clone https://github.com/mostafa-ja/malware_detection.git

In [ ]:
import gdown
# our sharable address : https://drive.google.com/file/d/1-MPYbRTFn_zH3rybfAYihHuIjq2XSiI5/view?usp=sharing
download_link = 'https://drive.google.com/uc?id=1-MPYbRTFn_zH3rybfAYihHuIjq2XSiI5'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1--gjFTkAJUW74RXWE4anJ5wssV7K_VAo'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1-2NnBk3YKauGTGK8DfQHBxrLrYyTxNsp'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1-4PhQGYHpMXrCcxcgMadgPPA4S-9yluX'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
download_link = 'https://drive.google.com/uc?id=1JmP8VxTUU3Yi7dle247PKlgrS6WS91x8'
output_filepath = '/content/'
gdown.download(download_link, output_filepath)

In [ ]:
!cp /content/train_dataset.pt /content/malware_detection/datasets
!cp /content/validation_dataset.pt /content/malware_detection/datasets
!cp /content/test_dataset.pt /content/malware_detection/datasets

In [ ]:
import torch
import torch.utils.data as data

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = [(torch.tensor(data, dtype=torch.float32), label) for data, label in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

In [ ]:
class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dense_hidden_units=[200, 200], dropout=0.6, smooth=False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.name = name

        self.dropout = dropout
        self.smooth = smooth
        self.dense_hidden_units = dense_hidden_units

        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))

        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu

        self.model_save_path = path.join('/content/malware_detection/save/drebin/md_dnn'+ '_' + self.name,'model.pth')

    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def customize_loss(self, logits, gt_labels, representation = None, mini_batch_idx=None):
        return F.cross_entropy(logits, gt_labels)

    # for predict pertubed malicious and create a y_true same size for theam
    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer: # for large dataset we have to consider test dataset with batches
                x, y = x.double().to(device), y.long().to(device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences) #[[1,2,3],[4,5,6]] > below each other
        gt_labels = torch.cat(gt_labels, dim=0) #[[1,2,3],[4,5,6]] > [1,2,3,4,5,6]
        return confidences, gt_labels


    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print("The accuracy on the test dataset is {:.5f}%".format(accuracy * 100))
        print("The balanced accuracy on the test dataset is {:.5f}%".format(b_accuracy * 100))


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print("False Negative Rate (FNR) is {:.5f}%, False Positive Rate (FPR) is {:.5f}%, F1 score is {:.5f}%".format(fnr * 100, fpr * 100, f1 * 100))

In [ ]:
train_dataset = torch.load('/content/malware_detection/datasets/train_dataset.pt')
train_Loader = data.DataLoader(train_dataset, batch_size=128, shuffle=False)
validation_dataset = torch.load('/content/malware_detection/datasets/validation_dataset.pt')
validation_Loader = data.DataLoader(validation_dataset, batch_size=10000, shuffle=False)
test_dataset = torch.load('/content/malware_detection/datasets/test_dataset.pt')
test_Loader = data.DataLoader(test_dataset, batch_size=10000, shuffle=False)

In [ ]:
import pickle
with open('/content/malware_detection/dataset/drebin/data.vocab', 'rb') as file:
    loaded_data = pickle.load(file)

print(len(loaded_data))

In [ ]:
model = MalwareDetectionDNN(6693,2)
model = model.double().to(device)

In [ ]:
model.load_state_dict(torch.load('/content/malware_detection/model.pth'))

In [ ]:
# SO IMPORTANT
model.eval()

## Step 2 — Probing the model: what the fitness function will read

In [ ]:
for X, y in train_Loader:
  break
print(X.shape)
print(y.shape)

In [ ]:
x = X[0]
print(x.shape)

In [ ]:
model(x.double())

In [ ]:
model.inference_batch_wise(x.double())

In [ ]:
# probability of being malware
model.inference_batch_wise(x.double())[1]

In [ ]:
F.cross_entropy(model(x.double()),  y[4].long())

## Step 3 — Building the pieces: selection probabilities and the changable mask

In [ ]:
from core.droidfeature import InverseDroidFeature

inverse_feature = InverseDroidFeature()

# manipulation_x : removable features
manipulation_x = inverse_feature.get_manipulation()
manipulation_x = torch.LongTensor(manipulation_x).to(device)

# index of vocab which are interdependent
omega = inverse_feature.get_interdependent_apis()
omega = torch.sum(F.one_hot(torch.tensor(omega), num_classes=len(inverse_feature.vocab)),dim=0).to(device)

# a list of vocab length : if the type of feature is api , index of that feature will be true
api_flag = inverse_feature.get_api_flag()

In [ ]:
# Assuming 'a' is a list of numbers
a = [1, 2, 4, 6, 5,9,2]

# Invert each element
b = [1 / x for x in a]

print(b*sum(a))

In [ ]:
fitness_values

In [ ]:
fitness_values = torch.tensor([1, 2, 4, 6, 5,9,2])
s = fitness_values.sum()
fitness_values = [s / x for x in fitness_values]

total_fitness = torch.sum(torch.tensor(fitness_values))

# Calculate selection probabilities
selection_probabilities = torch.tensor(fitness_values) / total_fitness

In [ ]:
selection_probabilities

In [ ]:
fitness_values = [1, 2, 4, 8, 16]
s = sum(fitness_values)
fitness_values = [s / x for x in fitness_values]
total_fitness = torch.sum(torch.tensor(fitness_values))

# Calculate selection probabilities
selection_probabilities = torch.tensor(fitness_values) / total_fitness

# Perform roulette wheel selection to choose parents
for _ in range(len(fitness_values)):
    rand_val = random.random()
    cumulative_prob = 0.0
    for i, prob in enumerate(selection_probabilities):
        cumulative_prob += prob.item()
        if rand_val <= cumulative_prob:
            print(i)
            break

In [ ]:
def changable(adv_features,manipulation_x):
  pos_insertion = (adv_features <= 0.5) * 1
  pos_removal = (adv_features > 0.5)& (manipulation_x) * 1
  mask = pos_insertion - pos_removal
  return mask

# Initialization Function
def initialization(original_example, manipulation_x, num_individuals,probability):
    initial_delta = []
    for _ in range(num_individuals):
        prob = torch.rand(original_example.size()) < probability
        change = changable(original_example,manipulation_x)
        delta = change * prob
        initial_delta.append(delta)
    return initial_delta

# Fitness Function
def fitness_function2(individual,delta, alpha):
  pertubation = torch.sum(abs(delta))
  pertubed_x = individual + delta

  fitness = 180 * model.inference_batch_wise(pertubed_x.double())[0][1] + alpha * pertubation

  return fitness

def fitness_function(individual,delta, alpha):
  pertubation = torch.sum(abs(delta))
  pertubed_x = individual + delta
  if model.inference_batch_wise(pertubed_x.double())[0].argmax() == 1 :
      fitness = 80 * (model.inference_batch_wise(pertubed_x.double())[0][1])
  else :
      fitness = model.inference_batch_wise(pertubed_x.double())[0][1] + alpha * pertubation

  return fitness

def selection(population, fitness_values):
    selected_parents = []
    s = sum(fitness_values)
    fitness_values = [s / x for x in fitness_values]
    total_fitness = torch.sum(torch.tensor(fitness_values))

    # Calculate selection probabilities
    selection_probabilities = torch.tensor(fitness_values) / total_fitness

    # Perform roulette wheel selection to choose parents
    for _ in range(len(population)):
        rand_val = random.random()
        cumulative_prob = 0.0
        for i, prob in enumerate(selection_probabilities):
            cumulative_prob += prob.item()
            if rand_val <= cumulative_prob:
                selected_parents.append(population[i])
                break

    return selected_parents


def selection2(population, fitness_values):
    selected_parents = []
    total_fitness = torch.sum(torch.tensor(fitness_values))

    # Calculate selection probabilities
    selection_probabilities = 1 - (torch.tensor(fitness_values) - torch.min(torch.tensor(fitness_values))) / total_fitness
    #print(selection_probabilities)
    # Perform roulette wheel selection to choose parents
    for _ in range(len(population)):
        rand_val = random.random()
        cumulative_prob = 0.0
        for i, prob in enumerate(selection_probabilities):
            cumulative_prob += prob.item()
            if rand_val <= cumulative_prob:
                selected_parents.append(population[i])
                break

    return selected_parents

def crossover(parent_1, parent_2, crossover_prob):

    # Create a matrix B of the same size as the parents filled with random 0s and 1s
    B = torch.randint(0, 2, size=parent_1.size()).float()
    if random.random() < crossover_prob :
      child_1 = parent_1 * B  + parent_2 * (1 - B)
      child_2 = parent_1 * (1 - B)  + parent_2 * B
    else :
      child_1 = parent_1
      child_2 = parent_2

    return child_1, child_2

# Crossover: Single-point crossover
def crossover2(parent_1, parent_2, crossover_prob):
    if random.random() < crossover_prob :
      crossover_point = random.randint(0, len(parent_1) - 1)
      child_1 = torch.cat((parent_1[:crossover_point], parent_2[crossover_point:]), dim=0)
      child_2 = torch.cat((parent_2[:crossover_point], parent_1[crossover_point:]), dim=0)
    else :
      child_1 = parent_1
      child_2 = parent_2
    return child_1, child_2

def crossover3(parent_1, parent_2, crossover_prob):

    # Create a matrix B of the same size as the parents filled with random 0s and 1s
    B = torch.randint(0, 2, size=parent_1.size()).float()

    # Perform crossover based on crossover probability and crossover matrix B
    mask = (torch.rand(parent_1.size()) < crossover_prob) * 1.
    inv_mask = 1. - mask

    child_1 = parent_1 * B * mask + parent_2 * (1 - B) * mask + parent_1 * inv_mask
    child_2 = parent_1 * (1 - B) * mask + parent_2 * B * mask + parent_2 * inv_mask

    return child_1, child_2

def mutation(individual, A_cross, mutation_prob,manipulation_x):
    prob = torch.rand(A_cross.size()) > mutation_prob
    mutated_A = A_cross * prob
    return mutated_A

def mutation2(individual, A_cross, mutation_prob,manipulation_x):

    change = changable(individual+A_cross,manipulation_x)
    prob = torch.rand(A_cross.size()) < mutation_prob
    mask = change*prob

    mutated_A = A_cross + mask

    return mutated_A

# Main Genetic Algorithm
def genetic_algorithm(original_example, manipulation_x, num_individuals, max_generation, gamma, alpha, initial_prob, crossover_prob, mutation_prob):
    population = initialization(original_example, manipulation_x, num_individuals,initial_prob)
    fitness = []
    fit_min = 100
    for t in range(max_generation):
        # Calculate fitness values for the current population
        fitness_values = [fitness_function(original_example, delta, alpha) for delta in population]
        fitness.append(torch.min(torch.tensor(fitness_values)))
        if torch.min(torch.tensor(fitness_values)) < fit_min :
            fit_min = torch.min(torch.tensor(fitness_values))
            best_adversarial_example = population[torch.argmin(torch.tensor(fitness_values))]
            print(torch.sum(abs(best_adversarial_example)))
            print(fit_min)
        # Check termination condition
        #if max(fitness_values) < gamma:
            #break

        # Selection
        selected_parents = selection(population, fitness_values)

        # Create new population using crossover and mutation
        new_population = []
        for i in range(0, num_individuals, 2):
            parent_1 = selected_parents[i]
            parent_2 = selected_parents[i + 1]

            child_1, child_2 = crossover(parent_1, parent_2, crossover_prob)
            child_1 = mutation(original_example, child_1, mutation_prob, manipulation_x)
            child_2 = mutation(original_example, child_2, mutation_prob, manipulation_x)

            new_population.extend([child_1, child_2])

        population = new_population

    # Return the best adversarial example after termination

    return best_adversarial_example, fitness

## Step 4 — The genetic algorithm, and the first run (200 individuals, 50 generations)



```
original_example = x
num_individuals = 200
max_generation = 50
gamma = 0.5
alpha = 0.1
initial_prob = 0.001
crossover_prob = 0.5
mutation_prob = 0.0001
```



In [ ]:
samples,labels = next(iter(test_Loader))
x,label = samples[0],labels[0]

In [ ]:
original_example = x
num_individuals = 200
max_generation = 50
gamma = 0.5
alpha = 0.2
initial_prob = 0.001
crossover_prob = 0.5
mutation_prob = 0.0001

best_adversarial_example, fitness = genetic_algorithm(original_example, manipulation_x, num_individuals, max_generation, gamma, alpha, initial_prob, crossover_prob, mutation_prob)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(fitness)

In [ ]:
pertubed_x = x + best_adversarial_example
model.inference_batch_wise(pertubed_x.double())[1]

In [ ]:
fitness_function(x,best_adversarial_example, alpha)

In [ ]:
torch.sum(abs(best_adversarial_example))

In [ ]:
x.sum()

## Step 5 — A bigger run (500 individuals)

In [ ]:
original_example = x[0]
num_individuals = 500
max_generation = 50
gamma = 0.5
alpha = 0.01
initial_prob = 0.1
crossover_prob = 0.5
mutation_prob = 0.1

best_adversarial_example, fitness = genetic_algorithm(original_example, manipulation_x, num_individuals, max_generation, gamma, alpha, initial_prob, crossover_prob, mutation_prob)

In [ ]:
model.inference_batch_wise((x[0]+best_adversarial_example).double())[0]

## Step 6 — Version 2: feature importance from saliency

In [ ]:
from captum.attr import Saliency

def importance(original_example):
  # Assuming x is your input data
  x_tensor = torch.tensor(x[0], requires_grad=True)
  model.eval()  # Set the model to evaluation mode

  # Forward pass
  output = model(x.double())

  # Get gradients
  saliency = Saliency(model)
  attributions = saliency.attribute(x.double(), target=0)  # Targeting class 0 for explanation

  # Normalize attributions if necessary
  attributions = attributions / torch.norm(attributions)

  return attributions

## Step 7 — Version 2: seeding the population by mimicry

#mimicry

In [ ]:
for X, y in test_Loader:
  break
print(X.shape)
print(y.shape)

In [ ]:
mal_x = X[y==1]
ben_x = X[y==0]

In [ ]:
trials = 1000
indices = torch.randperm(len(ben_x))[:trials]
trial_vectors = ben_x[indices]
x_fixed_one = ((1. - manipulation_x).float() *x)
modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
modified_x = modified_x.double()

In [ ]:
# Initialization Function
def initialization(original_example, ben_x, manipulation_x, num_individuals):
    indices = torch.randperm(len(ben_x))[:num_individuals]
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    return modified_x

In [ ]:
initialization(x, ben_x, manipulation_x, 10).shape

In [ ]:
original_example = mal_x[0:1]
num_individuals = 200
max_generation = 50
gamma = 0.5
alpha = 0.1
crossover_prob = 0.1
mutation_prob = 0.0001

best_adversarial_example, fitness = genetic_algorithm(original_example, manipulation_x, num_individuals, max_generation, gamma, alpha, crossover_prob, mutation_prob)

In [ ]:
pertubed_x = original_example + best_adversarial_example
model.inference_batch_wise(pertubed_x.double())[0][0][1]

In [ ]:
fitness_function(original_example,best_adversarial_example, alpha)

## Step 8 — Version 3: the operators, rebuilt (mutation, initialization, changable)

#mimicry

In [ ]:
original_example = torch.tensor([1,1,1,0,0,1])
delta = torch.tensor([-1,0,0,0,1,-1])
manipulation_x = torch.tensor([0,0,1,1,1,1]) #removable features
x_fixed_one = ((1. - manipulation_x).float() * original_example)
x_fixed_one

In [ ]:
prob = torch.rand(original_example.size()) < 0.7
prob

In [ ]:
add = prob * (1-(original_example+delta))
add

In [ ]:
remove = (prob * (original_example+delta) * manipulation_x) * -1.

remove

In [ ]:
def mutation(original_example, A_cross,fixed_one, mutation_prob):
  prob = torch.rand(original_example.size()) < mutation_prob
  add = (prob * (1-(original_example+A_cross))) * 1.
  remove = (prob * (original_example+A_cross) * manipulation_x) * 1.
  mutated_A = A_cross + add - remove
  return A_cross

In [ ]:
import torch

def mutation(A_cross,original_example,fixed_one, mutation_prob=0.1):
    # Calculate the number of elements to change based on the mutation_prob
    num_elements = int(mutation_prob * A_cross.numel())

    # Get random indices to change
    indices = torch.randperm(A_cross.numel())[:num_elements]

    # Generate random values to replace the existing values
    new_values = torch.randint(-1, 2, (num_elements,))

    # Update the values at randomly chosen indices
    A_cross.view(-1)[indices] = new_values
    mutated_A = A_cross * (1-fixed_one)
    return mutated_A

# Example: Create a 5x5 matrix with values 1, 0, -1
matrix = torch.randint(-1, 2, (1,10))
print(matrix)
# Change 10% of the values randomly among 1, 0, -1
changed_matrix = random_change_values(matrix, change_percentage=0.1)

print(changed_matrix)

In [ ]:
def initialization(original_example, ben_x, manipulation_x, num_individuals):
    indices = torch.randperm(len(ben_x))[:num_individuals]
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    delta = modified_x - original_example
    return delta, x_fixed_one

In [ ]:
def changable(adv_features,manipulation_x):
  pos_insertion = (adv_features <= 0.5) * 1
  pos_removal = (adv_features > 0.5)& (manipulation_x) * 1
  mask = pos_insertion - pos_removal
  return mask

# Initialization Function
def initialization(original_example, ben_x, manipulation_x, num_individuals):
    indices = torch.randperm(len(ben_x))[:num_individuals]
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    delta = modified_x - original_example
    return delta, x_fixed_one

def initialization2(original_example, ben_x, manipulation_x, num_individuals):
    indices = torch.randperm(len(ben_x))[:num_individuals]
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    delta = modified_x - original_example
    return delta


def fitness_function(individual,delta, alpha):
  pertubation = torch.sum(abs(delta))
  pertubed_x = individual + delta
  if model.inference_batch_wise(pertubed_x.double())[0][0].argmax() == 1 :
      fitness = 50 * (model.inference_batch_wise(pertubed_x.double())[0][0][1])
  else :
      fitness = model.inference_batch_wise(pertubed_x.double())[0][0][1] + alpha * pertubation

  return fitness

def selection(population, fitness_values):
    selected_parents = []
    s = sum(fitness_values)
    fitness_values = [s / x for x in fitness_values]
    total_fitness = torch.sum(torch.tensor(fitness_values))

    # Calculate selection probabilities
    selection_probabilities = torch.tensor(fitness_values) / total_fitness

    # Perform roulette wheel selection to choose parents
    for _ in range(len(population)):
        rand_val = random.random()
        cumulative_prob = 0.0
        for i, prob in enumerate(selection_probabilities):
            cumulative_prob += prob.item()
            if rand_val <= cumulative_prob:
                selected_parents.append(population[i])
                break

    return selected_parents



def crossover(parent_1, parent_2, crossover_prob):

    # Create a matrix B of the same size as the parents filled with random 0s and 1s
    B = torch.randint(0, 2, size=parent_1.size()).float()
    if random.random() < crossover_prob :
      child_1 = parent_1 * B  + parent_2 * (1 - B)
      child_2 = parent_1 * (1 - B)  + parent_2 * B
    else :
      child_1 = parent_1
      child_2 = parent_2

    return child_1, child_2

def mutation(original_example, A_cross,fixed_one, mutation_prob):
  prob = torch.rand(original_example.size()) < mutation_prob
  add = (prob * (1-(original_example+A_cross))) * 1.
  remove = (prob * (original_example+A_cross) * manipulation_x) * 1.
  mutated_A = A_cross + add - remove
  return A_cross

def mutation3(A_cross,fixed_one, mutation_prob=0.1):
    # Calculate the number of elements to change based on the mutation_prob
    num_elements = int(mutation_prob * A_cross.numel())

    # Get random indices to change
    indices = torch.randperm(A_cross.numel())[:num_elements]

    # Generate random values to replace the existing values
    new_values = torch.randint(0, 2, (num_elements,), dtype=A_cross.dtype)

    # Update the values at randomly chosen indices
    A_cross.view(-1)[indices] = new_values
    mutated_A = A_cross * (1-fixed_one)
    return mutated_A

def mutation2(individual, A_cross, mutation_prob,manipulation_x):
    prob = torch.rand(A_cross.size()) < mutation_prob
    mutated_A = A_cross * prob
    return mutated_A


# Main Genetic Algorithm
# Main Genetic Algorithm
def genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob):
    population, fixed_one = initialization(original_example, ben_x, manipulation_x, num_individuals)
    fitness = []
    fit_min = 100
    for t in range(max_generation):
        # Calculate fitness values for the current population
        fitness_values = [fitness_function(original_example, delta, alpha) for delta in population]
        fitness.append(torch.min(torch.tensor(fitness_values)))
        if torch.min(torch.tensor(fitness_values)) < fit_min :
            fit_min = torch.min(torch.tensor(fitness_values))
            print('t = ',t)
            best_adversarial_example = population[torch.argmin(torch.tensor(fitness_values))]
            print(torch.sum(abs(best_adversarial_example)))
            print(fit_min)
        # Check termination condition
        #if max(fitness_values) < gamma:
            #break

        # Selection
        selected_parents = selection(population, fitness_values)

        # Create new population using crossover and mutation
        new_population = []
        for i in range(0, num_individuals, 2):
            parent_1 = selected_parents[i]
            parent_2 = selected_parents[i + 1]

            child_1, child_2 = crossover(parent_1, parent_2, crossover_prob)
            child_1 = mutation(original_example,child_1, fixed_one, mutation_prob)
            child_2 = mutation(original_example,child_2, fixed_one, mutation_prob)

            new_population.extend([child_1, child_2])

        population = new_population

    # Return the best adversarial example after termination

    return best_adversarial_example, fitness

def genetic_algorithm2(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob):
    population = initialization(original_example, ben_x, manipulation_x, num_individuals)
    fitness = []
    fit_min = 100
    for t in range(max_generation):
        # Calculate fitness values for the current population
        fitness_values = [fitness_function(original_example, delta, alpha) for delta in population]
        fitness.append(torch.min(torch.tensor(fitness_values)))
        if torch.min(torch.tensor(fitness_values)) < fit_min :
            fit_min = torch.min(torch.tensor(fitness_values))
            best_adversarial_example = population[torch.argmin(torch.tensor(fitness_values))]
            print(torch.sum(abs(best_adversarial_example)))
            print(fit_min)
        # Check termination condition
        #if max(fitness_values) < gamma:
            #break

        # Selection
        selected_parents = selection(population, fitness_values)

        # Create new population using crossover and mutation
        new_population = []
        for i in range(0, num_individuals, 2):
            parent_1 = selected_parents[i]
            parent_2 = selected_parents[i + 1]

            child_1, child_2 = crossover(parent_1, parent_2, crossover_prob)
            child_1 = mutation(original_example, child_1, mutation_prob, manipulation_x)
            child_2 = mutation(original_example, child_2, mutation_prob, manipulation_x)

            new_population.extend([child_1, child_2])

        population = new_population

    # Return the best adversarial example after termination

    return best_adversarial_example, fitness

## Step 9 — Version 3: runs against individual malware samples

In [ ]:
model = MalwareDetectionDNN(6693,2)
model = model.double().to(device)
ckpt = torch.load('/content/md_at_pgd.pth')
model.load_state_dict(ckpt['model'])

In [ ]:
X,y = next(iter(test_Loader))
mal_x = X[y==1]
ben_x = X[y==0]

In [ ]:
len(mal_x)

In [ ]:
i = 7
original_example = mal_x[i:i+1]
num_individuals = 100
max_generation = 10
alpha = 0.01
crossover_prob = 0.5
mutation_prob = 0.005

best_adversarial_example, fitness = genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob)

In [ ]:
model(original_example.double())

In [ ]:
model(best_adversarial_example.double())

In [ ]:
model.inference_batch_wise((original_example).double())[0]

In [ ]:
model.inference_batch_wise((original_example+best_adversarial_example).double())[0]

In [ ]:
torch.sum(best_adversarial_example)

In [ ]:
torch.sum(torch.abs(best_adversarial_example))

In [ ]:
torch.sum(original_example)

In [ ]:
torch.sum(original_example+best_adversarial_example)

## Step 10 — Attacking every malware sample in the batch

In [ ]:
attack = 0
for i in range(len(mal_x)):
  print('------------------------------')
  print('i = ',i)
  original_example = mal_x[i:i+1]
  num_individuals = 400
  max_generation = 10
  alpha = 0.01
  crossover_prob = 0.5
  mutation_prob = 0.005

  best_adversarial_example, fitness = genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob)
  if fitness[-1] < 10:
      attack += 1
      print('from ',i+1,' samples, we attacked ',attack)

## Step 11 — Guiding evolution with RandomForest feature importances

#Extract important features

In [ ]:
train_Loader = data.DataLoader(train_dataset, batch_size=10000, shuffle=False)
X_train,y_train = next(iter(train_Loader))
X_train.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a RandomForestClassifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model
clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

X_test,y_test = next(iter(test_Loader))

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
feature_importances = clf.feature_importances_

print(sum(feature_importances))

In [ ]:
feature_importances2 = torch.tensor(feature_importances**(0.2))

In [ ]:
i = 4
original_example = mal_x[i:i+1]
num_individuals = 100
max_generation = 100
alpha = 0.01
crossover_prob = 0.5
mutation_prob = 0.1

best_adversarial_example, fitness = genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob)

## Step 12 — A better initial population: the nearest benign samples

# new initial population


In [ ]:
i=1
original_example = mal_x[i:i+1]

In [ ]:
diff = torch.abs(original_example - ben_x)

In [ ]:
torch.argmin(torch.sum(diff,dim=1))

In [ ]:
torch.topk(torch.sum(diff,dim=1), k=10, largest=False).indices

In [ ]:
# Initialization Function
def initialization(original_example, ben_x, manipulation_x, num_individuals):
    diff = torch.abs(original_example - ben_x)
    indices = torch.topk(torch.sum(diff,dim=1), k=num_individuals, largest=False).indices
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    delta = modified_x - original_example
    return delta, x_fixed_one

def initialization2(original_example, ben_x, manipulation_x, num_individuals):
    indices = torch.randperm(len(ben_x))[:num_individuals]
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    delta = modified_x - original_example
    return delta, x_fixed_one

def fitness_function(individual,delta, alpha):
  pertubation = torch.sum(abs(delta))
  pertubed_x = individual + delta
  if model.inference_batch_wise(pertubed_x.double())[0][0].argmax() == 1 :
      fitness = 50 * (model.inference_batch_wise(pertubed_x.double())[0][0][1])
  else :
      fitness = model.inference_batch_wise(pertubed_x.double())[0][0][1] + alpha * pertubation

  return fitness

def selection(population, fitness_values):
    selected_parents = []
    s = sum(fitness_values)
    fitness_values = [s / x for x in fitness_values]
    total_fitness = torch.sum(torch.tensor(fitness_values))

    # Calculate selection probabilities
    selection_probabilities = torch.tensor(fitness_values) / total_fitness

    # Perform roulette wheel selection to choose parents
    for _ in range(len(population)):
        rand_val = random.random()
        cumulative_prob = 0.0
        for i, prob in enumerate(selection_probabilities):
            cumulative_prob += prob.item()
            if rand_val <= cumulative_prob:
                selected_parents.append(population[i])
                break

    return selected_parents



def crossover(parent_1, parent_2, crossover_prob):

    # Create a matrix B of the same size as the parents filled with random 0s and 1s
    B = torch.randint(0, 2, size=parent_1.size()).float()
    if random.random() < crossover_prob :
      child_1 = parent_1 * B  + parent_2 * (1 - B)
      child_2 = parent_1 * (1 - B)  + parent_2 * B
    else :
      child_1 = parent_1
      child_2 = parent_2

    return child_1, child_2



def mutation(original_example, A_cross,fixed_one, mutation_prob):
  prob = torch.rand(original_example.size()) < mutation_prob
  add = (prob * (1-(original_example+A_cross))) * 1.
  remove = (prob * (original_example+A_cross) * manipulation_x) * 1.
  mutated_A = A_cross + add - remove
  return A_cross

def mutation3(A_cross,fixed_one, mutation_prob=0.1):
    # Calculate the number of elements to change based on the mutation_prob
    num_elements = int(mutation_prob * A_cross.numel())

    # Get random indices to change
    indices = torch.randperm(A_cross.numel())[:num_elements]

    # Generate random values to replace the existing values
    new_values = torch.randint(0, 2, (num_elements,), dtype=A_cross.dtype)

    # Update the values at randomly chosen indices
    A_cross.view(-1)[indices] = new_values
    mutated_A = A_cross * (1-fixed_one)
    return mutated_A

def mutation2(individual, A_cross, mutation_prob,manipulation_x):
    prob = torch.rand(A_cross.size()) < mutation_prob
    mutated_A = A_cross * prob
    return mutated_A


# Main Genetic Algorithm
# Main Genetic Algorithm
def genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob):
    population, fixed_one = initialization(original_example, ben_x, manipulation_x, num_individuals)
    fitness = []
    fit_min = 100
    for t in range(max_generation):
        # Calculate fitness values for the current population
        fitness_values = [fitness_function(original_example, delta, alpha) for delta in population]
        fitness.append(torch.min(torch.tensor(fitness_values)))
        if torch.min(torch.tensor(fitness_values)) < fit_min :
            fit_min = torch.min(torch.tensor(fitness_values))
            print('t = ',t)
            best_adversarial_example = population[torch.argmin(torch.tensor(fitness_values))]
            print(torch.sum(abs(best_adversarial_example)))
            print(fit_min)
        # Check termination condition
        #if max(fitness_values) < gamma:
            #break

        # Selection
        selected_parents = selection(population, fitness_values)

        # Create new population using crossover and mutation
        new_population = []
        for i in range(0, num_individuals, 2):
            parent_1 = selected_parents[i]
            parent_2 = selected_parents[i + 1]

            child_1, child_2 = crossover(parent_1, parent_2, crossover_prob)
            child_1 = mutation(original_example,child_1, fixed_one, mutation_prob)
            child_2 = mutation(original_example,child_2, fixed_one, mutation_prob)

            new_population.extend([child_1, child_2])

        population = new_population

    # Return the best adversarial example after termination

    return best_adversarial_example, fitness


def genetic_algorithm2(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob):
    population, fixed_one = initialization2(original_example, ben_x, manipulation_x, num_individuals)
    fitness = []
    fit_min = 100
    for t in range(max_generation):
        # Calculate fitness values for the current population
        fitness_values = [fitness_function(original_example, delta, alpha) for delta in population]
        fitness.append(torch.min(torch.tensor(fitness_values)))
        if torch.min(torch.tensor(fitness_values)) < fit_min :
            fit_min = torch.min(torch.tensor(fitness_values))
            print('t = ',t)
            best_adversarial_example = population[torch.argmin(torch.tensor(fitness_values))]
            print(torch.sum(abs(best_adversarial_example)))
            print(fit_min)
        # Check termination condition
        #if max(fitness_values) < gamma:
            #break

        # Selection
        selected_parents = selection(population, fitness_values)

        # Create new population using crossover and mutation
        new_population = []
        for i in range(0, num_individuals, 2):
            parent_1 = selected_parents[i]
            parent_2 = selected_parents[i + 1]

            child_1, child_2 = crossover(parent_1, parent_2, crossover_prob)
            child_1 = mutation(original_example,child_1, fixed_one, mutation_prob)
            child_2 = mutation(original_example,child_2, fixed_one, mutation_prob)

            new_population.extend([child_1, child_2])

        population = new_population

    # Return the best adversarial example after termination

    return best_adversarial_example, fitness

In [ ]:
i = 4
original_example = mal_x[i:i+1]
num_individuals = 20
max_generation = 50
alpha = 0.01
crossover_prob = 0.5
mutation_prob = 0.1

best_adversarial_example, fitness = genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob)

In [ ]:
# we use benign training data with least difference with our example in initial population
num_individuals = 400
max_generation = 10
alpha = 0.01
crossover_prob = 0.5
mutation_prob = 0.1
success = 0

for i in range(100):
  print('*****************************')
  print('i = ',i)
  original_example = mal_x[i:i+1]
  best_adversarial_example, fitness = genetic_algorithm(original_example, ben_x_train, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob)
  if torch.min(torch.tensor(fitness)) < 40:
    success += 1

print(success)

In [ ]:
print(success)

## Step 13 — Weighted mutation, in the mimicry setting

#  Mimicry Attacks
Under this scenario, the attacker is assumed to be able to
collect a surrogate dataset including malware and benign
samples, and to know the feature space. Accordingly, θ =
(Dˆ, X ). In this case, the attack strategy amounts to manipulating malware samples to make them as close as possible to
the benign data (in terms of conditional probability distributions or, alternatively, distance in feature space). To this end,
in the case of Drebin (which uses binary feature values), we
can assume that the attacker still aims to minimize Eq. (4),
but estimates each component of wˆ independently for each
feature as wˆk = p(ˆxk = 1|y = +1) − p(ˆxk = 1|y = −1),
k = 1, . . . , d. This will indeed induce the attacker to add
(remove) first features which are more frequently present
(absent) in benign files, making the probability distribution
of malware samples closer to that of the benign data.

In [ ]:
import torch

# Function to estimate w_k for each feature
def estimate_weights(malware_samples, benign_samples):
    # Convert arrays to PyTorch tensors
    malware_samples_tensor = torch.tensor(malware_samples, dtype=torch.float32)
    benign_samples_tensor = torch.tensor(benign_samples, dtype=torch.float32)

    # Calculate conditional probabilities for each feature
    p_malware = torch.mean(malware_samples_tensor, dim=0)  # Probability of feature being 1 in malware samples
    p_benign = torch.mean(benign_samples_tensor, dim=0)    # Probability of feature being 1 in benign samples

    # Estimate w_k for each feature
    weights = p_benign - p_malware

    return weights

# Example datasets (replace with your actual datasets)
malware_samples = [
    [1, 0, 1, 0],  # Replace with your malware samples
    [0, 1, 0, 1],
    [1, 1, 0, 0],
]

benign_samples = [
    [0, 1, 1, 0],  # Replace with your benign samples
    [1, 0, 1, 1],
    [0, 1, 0, 0],
]

# Estimate weights for each feature
weights = estimate_weights(malware_samples, benign_samples)

print("Estimated weights for each feature:")
print(weights)

In [ ]:
# Function to estimate w_k for each feature
def estimate_weights(malware_samples, benign_samples):

    # Calculate conditional probabilities for each feature
    p_malware = torch.mean(malware_samples, dim=0)  # Probability of feature being 1 in malware samples
    p_benign = torch.mean(benign_samples, dim=0)    # Probability of feature being 1 in benign samples

    # Estimate w_k for each feature
    weights = p_benign - p_malware

    return weights

# Estimate weights for each feature
weights = estimate_weights(mal_x_train, ben_x_train)

print("Estimated weights for each feature:")
print(weights)

In [ ]:
weights.max()

In [ ]:
import torch
import random

# Function to perform feature mutation based on weights
def mutate_features(data_samples, weights):
    mutated_samples = torch.tensor(data_samples, dtype=torch.float32)

    # Generate random numbers for mutation probability
    mutation_probs = torch.rand(weights.shape)

    # For negative weights, remove features
    remove_indices = torch.where(weights < 0)[0]
    for idx in remove_indices:
        mutation_mask = (mutation_probs[idx] < abs(weights[idx]))*1
        print(mutation_mask)
        mutated_samples[:, idx] *= (1 - mutation_mask)

    # For positive weights, add features
    add_indices = torch.where(weights > 0)[0]
    for idx in add_indices:
        mutation_mask = mutation_probs[idx] < weights[idx]
        mutated_samples[:, idx] += mutation_mask

    return mutated_samples

# Example datasets (replace with your actual datasets)
data_samples = [
    [1, 0, 1, 0],  # Replace with your data samples
    [0, 1, 0, 1],
    [1, 1, 0, 0],
]

# Example weights (replace with your actual estimated weights)
example_weights = torch.tensor([0.2, -0.3, 0.1, -0.2], dtype=torch.float32)

# Mutate features based on weights
mutated_data = mutate_features(data_samples, example_weights)

print("Mutated data samples:")
print(mutated_data)

In [ ]:
def mutation(original_example, A_cross,fixed_one):

  # Create a matrix where positive values stay and negative values become zero
  weights_pos = torch.clamp(weights, min=0)

  # Create a matrix where negative values stay and positive values become zero
  weights_neg = torch.clamp(weights, max=0)

  # Generate random numbers for mutation probability
  mutation_probs = torch.rand(weights.shape)

  prob_pos = weights_pos > mutation_prob
  prob_neg = weights_neg > mutation_prob
  add = (prob_pos * (1-(original_example+A_cross))) * 1.
  remove = (prob_neg * (original_example+A_cross) * manipulation_x) * 1.
  mutated_A = A_cross + add - remove
  return A_cross

In [ ]:
# Initialization Function
def initialization(original_example, ben_x, manipulation_x, num_individuals):
    diff = torch.abs(original_example - ben_x)
    indices = torch.topk(torch.sum(diff,dim=1), k=num_individuals, largest=False).indices
    trial_vectors = ben_x[indices]
    x_fixed_one = ((1. - manipulation_x).float() * original_example)
    modified_x = torch.clamp(x_fixed_one + trial_vectors, min=0., max=1.)
    modified_x = modified_x.double()
    delta = modified_x - original_example
    return delta, x_fixed_one

def fitness_function(individual,delta, alpha):
  pertubation = torch.sum(abs(delta))
  pertubed_x = individual + delta
  if model.inference_batch_wise(pertubed_x.double())[0][0].argmax() == 1 :
      fitness = 50 * (model.inference_batch_wise(pertubed_x.double())[0][0][1])
  else :
      fitness = model.inference_batch_wise(pertubed_x.double())[0][0][1] + alpha * pertubation

  return fitness

def selection(population, fitness_values):
    selected_parents = []
    s = sum(fitness_values)
    fitness_values = [s / x for x in fitness_values]
    total_fitness = torch.sum(torch.tensor(fitness_values))

    # Calculate selection probabilities
    selection_probabilities = torch.tensor(fitness_values) / total_fitness

    # Perform roulette wheel selection to choose parents
    for _ in range(len(population)):
        rand_val = random.random()
        cumulative_prob = 0.0
        for i, prob in enumerate(selection_probabilities):
            cumulative_prob += prob.item()
            if rand_val <= cumulative_prob:
                selected_parents.append(population[i])
                break

    return selected_parents



def crossover(parent_1, parent_2, crossover_prob):

    # Create a matrix B of the same size as the parents filled with random 0s and 1s
    B = torch.randint(0, 2, size=parent_1.size()).float()
    if random.random() < crossover_prob :
      child_1 = parent_1 * B  + parent_2 * (1 - B)
      child_2 = parent_1 * (1 - B)  + parent_2 * B
    else :
      child_1 = parent_1
      child_2 = parent_2

    return child_1, child_2



def mutation2(original_example, A_cross,fixed_one, mutation_prob):
  prob = torch.rand(original_example.size()) < mutation_prob
  add = (prob * (1-(original_example+A_cross))) * 1.
  remove = (prob * (original_example+A_cross) * manipulation_x) * 1.
  mutated_A = A_cross + add - remove
  return A_cross

def mutation(original_example, A_cross,fixed_one):

  # Create a matrix where positive values stay and negative values become zero
  weights_pos = torch.clamp(weights, min=0)

  # Create a matrix where negative values stay and positive values become zero
  weights_neg = torch.clamp(weights, max=0)

  # Generate random numbers for mutation probability
  mutation_probs = torch.rand(weights.shape)

  prob_pos = weights_pos > (mutation_probs/4.)
  #print(torch.sum(prob_pos))
  prob_neg = weights_neg > (mutation_probs/4.)
  #print(torch.sum(prob_neg))
  add = (prob_pos * (1-(original_example+A_cross))) * 1.
  remove = (prob_neg * (original_example+A_cross) * manipulation_x) * 1.
  mutated_A = A_cross + add - remove
  return A_cross

def mutation2(individual, A_cross, mutation_prob,manipulation_x):
    prob = torch.rand(A_cross.size()) < mutation_prob
    mutated_A = A_cross * prob
    return mutated_A


# Main Genetic Algorithm
# Main Genetic Algorithm
def genetic_algorithm(original_example, ben_x, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob):
    population, fixed_one = initialization(original_example, ben_x, manipulation_x, num_individuals)
    fitness = []
    fit_min = 100
    for t in range(max_generation):
        # Calculate fitness values for the current population
        fitness_values = [fitness_function(original_example, delta, alpha) for delta in population]
        fitness.append(torch.min(torch.tensor(fitness_values)))
        if torch.min(torch.tensor(fitness_values)) < fit_min :
            fit_min = torch.min(torch.tensor(fitness_values))
            print('t = ',t)
            best_adversarial_example = population[torch.argmin(torch.tensor(fitness_values))]
            print(torch.sum(abs(best_adversarial_example)))
            print(fit_min)
        # Check termination condition
        #if max(fitness_values) < gamma:
            #break

        # Selection
        selected_parents = selection(population, fitness_values)

        # Create new population using crossover and mutation
        new_population = []
        for i in range(0, num_individuals, 2):
            parent_1 = selected_parents[i]
            parent_2 = selected_parents[i + 1]

            child_1, child_2 = crossover(parent_1, parent_2, crossover_prob)
            child_1 = mutation(original_example,child_1, fixed_one)
            child_2 = mutation(original_example,child_2, fixed_one)

            new_population.extend([child_1, child_2])

        population = new_population

    # Return the best adversarial example after termination

    return best_adversarial_example, fitness

In [ ]:
i = 6
original_example = mal_x[i:i+1]
num_individuals = 100
max_generation = 100
alpha = 0.01
crossover_prob = 0.5
mutation_prob = 0.1

best_adversarial_example, fitness = genetic_algorithm(original_example, ben_x_train, manipulation_x, num_individuals, max_generation, alpha, crossover_prob, mutation_prob)